# 07 — Document Chunking Basics

## Objective

Understand why long documents must be broken into smaller pieces before they can be embedded, searched, or handed to an LLM as context -- and see, concretely, how the *strategy* used to do that breaking changes what each resulting piece actually contains. This notebook implements four chunking strategies from scratch (no framework) against one longer synthetic document, then applies the strategy it settles on to the 15 documents from Notebook 02, producing a `document_chunks` table that Notebook 08 builds on directly.

## What We Will Learn

- Why a whole document is usually the wrong unit to embed or retrieve, and what breaks if chunking is skipped entirely
- How naive fixed-size chunking works, and where it visibly goes wrong (cutting words and sentences in half)
- How overlap trades duplicate content for preserved context across a chunk boundary
- How recursive (structure-aware) chunking uses paragraph and sentence boundaries first, and only falls back to a hard character cut when a single paragraph is still too big
- Why "semantic chunking" is often approximated by structural boundaries (headings) when no embedding model is available yet, and what the real thing needs instead
- How chunk size and overlap change the number, size, and boundary quality of the resulting chunks, on the same source text

## Prerequisites

- Completed `02_synthetic_data_generation.ipynb` with matching catalog/schema widget values -- Step 8 below reads the `documents` Delta table it created and writes a new `document_chunks` table alongside it
- Basic Python (string slicing, list comprehensions, simple recursion)
- No Databricks AI Functions are used in this notebook -- every chunking function here is plain Python, so unlike Notebooks 04-06 nothing in the Implementation section is entitlement- or region-gated. The only Databricks-specific pieces are loading and writing Delta tables in Step 1 and Step 8

## Conceptual Explanation

**Why chunking is necessary.** Embedding models and LLM context windows are both bounded, and a retrieval system returns whole chunks -- never a hand-picked sentence out of a larger document. That makes the chunk the actual unit of retrieval granularity. A chunk that's too big dilutes its own embedding with content unrelated to what a query is asking about, and wastes LLM context budget on irrelevant text once retrieved. A chunk that's too small may not carry enough context to be useful on its own -- "This must be reported to the branch manager the same day it is discovered" means little without knowing what "this" refers to. The rest of this notebook makes that tension concrete instead of just describing it.

**Fixed-size chunking.** The simplest possible strategy: cut the text every `chunk_size` characters, no exceptions. It's fast, deterministic, and trivial to implement -- and it is completely blind to structure, because it never looks at the text itself, only counts characters. That blindness means it will just as happily cut through the middle of a word, a sentence, or a number ("10,000 units" could become "10,0" and "00 units") as through a natural break.

**Overlap.** Fixed-size chunking with a sliding window instead of adjacent, non-overlapping windows: chunk N+1 repeats the last `chunk_overlap` characters of chunk N. A sentence that got cut in half at a boundary is now likely to appear whole in at least one of the two neighboring chunks. Overlap does not fix the "blind to structure" problem -- it just makes the damage from that blindness less likely to matter, at the direct cost of storing (and later embedding) the same text more than once.

**Recursive / structure-aware chunking.** Instead of only counting characters, try splitting on the largest structural boundary first (paragraph breaks), and only fall back to a smaller boundary (sentence breaks, then raw characters) for any piece that is still too big after the previous attempt. This means most chunk boundaries land on a natural break -- but the character-level fallback still exists, and is identical to Step 3's naive chunking, because a single very long paragraph would otherwise never get split at all. This is essentially the algorithm behind LangChain's `RecursiveCharacterTextSplitter`, implemented directly here so the mechanics stay visible instead of hidden inside a library call.

**Semantic chunking.** The real version decides split points using an embedding model: it embeds consecutive sentences and starts a new chunk wherever the similarity between neighboring sentences drops, i.e. wherever the topic actually shifts. That requires an embedding model, which this project doesn't introduce until `09_embeddings_basics.ipynb`. This notebook uses a structural proxy instead -- splitting on the document's own headings, on the assumption that whoever wrote the document already marked its topic boundaries. That's a reasonable stand-in when a document has explicit headings, but it is not semantic chunking: it cannot find a topic shift inside a document with no headings, and a heading-bounded section can still be far too big or far too small to be a single good chunk (see Step 6's output).

**Chunk size trade-offs and context preservation.** Small chunks retrieve precisely but carry little standalone context. Large chunks carry more context but are more likely to mix multiple topics, which dilutes how well the chunk matches any single query, and pushes closer to the embedding model's or LLM's input limits. There is no universally correct chunk size -- it depends on the source documents' natural structure, the retrieval model's context window, and the LLM's context budget. Notebook 08 revisits this exact trade-off when it distinguishes `chunk_to_embed` from `chunk_to_retrieve`.

## Example Data

This notebook's main example is a single longer synthetic document written directly in Step 2 below: a five-section "Aurora Trust Bank -- Branch Operations Manual" (about 3,200 characters). It's long enough that a 400-character chunk size produces several chunks per section -- the 15 documents from `02_synthetic_data_generation.ipynb` are each a single short paragraph (roughly 500-700 characters), too short on their own to show a strategy's behavior across multiple chunks. Step 8 then applies the strategy chosen in Steps 2-7 to all 15 of Notebook 02's documents, producing a reusable `document_chunks` table.

## Implementation

### Step 1 — Parameterize catalog, schema, chunk size, and overlap

`chunk_size` and `chunk_overlap` are widgets, not hardcoded constants, specifically so the Experimentation Section can be run by changing a widget value and re-running the notebook -- no code edits required.

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")
dbutils.widgets.text("chunk_size", "400", "Chunk size (characters)")
dbutils.widgets.text("chunk_overlap", "60", "Overlap size (characters)")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
chunk_size = int(dbutils.widgets.get("chunk_size"))
chunk_overlap = int(dbutils.widgets.get("chunk_overlap"))

documents_table = f"{catalog_name}.{schema_name}.documents"
chunks_table = f"{catalog_name}.{schema_name}.document_chunks"

print(f"Documents table:             {documents_table}")
print(f"Chunks table (Step 8 output): {chunks_table}")
print(f"chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")

### Step 2 — Define one longer synthetic document

A five-section branch operations manual, hand-written and fictional (Aurora Trust Bank does not exist), with `## Heading` markers and blank-line-separated paragraphs -- deliberately shaped so that Steps 3-6 have real paragraph and heading boundaries to either respect or ignore.

In [ ]:
manual_text = (
    "Aurora Trust Bank -- Branch Operations Manual (Excerpt)\n\n"
    "## Opening Procedures\n\n"
    "Every branch must complete the opening checklist before serving customers each business "
    "day. This includes verifying that the vault time lock has released, counting and recording "
    "the opening cash position for each teller drawer, and confirming that the branch's core "
    "banking terminals are online and able to reach the central ledger. Any discrepancy between "
    "the previous night's closing balance and the current opening balance must be investigated "
    "before the first customer is served. Branches that cannot reconcile an opening discrepancy "
    "within thirty minutes must escalate to Regional Operations and may delay opening with "
    "regional approval.\n\n"
    "## Cash Handling\n\n"
    "Cash deposits above 5,000 units require dual counting by two tellers before the transaction "
    "is posted to the customer's account. Both tellers must independently arrive at the same "
    "total before the deposit is accepted; a mismatch is treated as a handling exception and is "
    "recounted from scratch rather than reconciled by adjustment. All teller cash drawers are "
    "reconciled at the end of each shift, not only at end of day, so that a shortage or overage "
    "is caught close to when it occurred. Any discrepancy over 50 units must be logged in the "
    "branch incident register and reported to the branch manager the same day it is discovered.\n\n"
    "## Wire Transfer Desk\n\n"
    "Outbound wire transfers are verified against the sender's registered signature before being "
    "queued for release. Transfers above 10,000 units additionally require a callback "
    "confirmation to a phone number already on file for the account, never a number supplied at "
    "the time of the transfer request. Wires are processed in two daily batches, at 11:00 and "
    "15:00, and the cutoff for inclusion in the afternoon batch is 14:30; requests submitted "
    "after cutoff are held for the following business day's morning batch. Any transfer flagged "
    "by the automated sanctions screening system is held for manual compliance review before "
    "release, regardless of amount.\n\n"
    "## End of Day Reconciliation\n\n"
    "Each branch reconciles teller cash positions, ATM cash levels, and the day's transaction "
    "log before close of business. The reconciliation compares the system-recorded totals "
    "against the physically counted cash, and any unresolved discrepancy is escalated to "
    "Regional Operations within one business day rather than carried forward silently. The "
    "completed reconciliation report is retained for seven years to satisfy the bank's record "
    "retention policy, and is subject to review during internal audit and regulatory "
    "examination.\n\n"
    "## Escalation and Incident Reporting\n\n"
    "Any incident involving a cash discrepancy, a suspected fraud indicator, or a failed wire "
    "callback is logged in the branch incident register at the time it is discovered, not at end "
    "of day. Incidents are classified by severity: a Tier 1 incident is resolved at branch level "
    "and reported to the branch manager; a Tier 2 incident is escalated to Regional Operations "
    "within one business day; a Tier 3 incident, such as suspected fraud or a compliance "
    "screening hit, is escalated immediately to both Regional Operations and the Compliance "
    "department, regardless of the amount involved.\n"
)

print(f"manual_text length: {len(manual_text)} characters")
print(manual_text)

### Step 3 — Naive fixed-size chunking

Cut the text every `chunk_size` characters. No look at word or sentence boundaries at all -- `show_chunks` (defined here, reused in the following steps) prints every chunk plus a specific look at the boundary between chunk 0 and chunk 1, so a mid-word cut is visible rather than just asserted.

In [ ]:
def chunk_fixed_size(text, size):
    """Cut text into non-overlapping windows of `size` characters. Blind to words, sentences, or structure."""
    return [text[i:i + size] for i in range(0, len(text), size)]


def show_chunks(chunks, label):
    lengths = [len(c) for c in chunks]
    print(f"{label}: {len(chunks)} chunks, sizes={lengths}")
    for i, c in enumerate(chunks):
        print(f"\n--- chunk {i} ({len(c)} chars) ---")
        print(c)


chunks_fixed = chunk_fixed_size(manual_text, chunk_size)
show_chunks(chunks_fixed, "FIXED-SIZE")

print("\n=== boundary between chunk 0 and chunk 1 ('|' marks the cut) ===")
print(chunks_fixed[0][-20:] + "|" + chunks_fixed[1][:20])

### Step 4 — Fixed-size chunking with overlap

Same idea, but the window slides forward by `chunk_size - chunk_overlap` instead of `chunk_size` each time, so consecutive chunks share their last/first `chunk_overlap` characters. The two printed snippets below should match exactly -- that shared text is the whole point of overlap.

In [ ]:
def chunk_fixed_size_overlap(text, size, overlap):
    """Like chunk_fixed_size, but each window shares `overlap` trailing characters with the next one."""
    assert overlap < size, "chunk_overlap must be smaller than chunk_size"
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += size - overlap
    return chunks


chunks_overlap = chunk_fixed_size_overlap(manual_text, chunk_size, chunk_overlap)
show_chunks(chunks_overlap, "FIXED-SIZE + OVERLAP")

print("\n=== shared text between chunk 0 and chunk 1 (should be identical) ===")
print("tail of chunk 0:", repr(chunks_overlap[0][-chunk_overlap:]))
print("head of chunk 1:", repr(chunks_overlap[1][:chunk_overlap]))
print("match:", chunks_overlap[0][-chunk_overlap:] == chunks_overlap[1][:chunk_overlap])

### Step 5 — Recursive (structure-aware) chunking

Try to split on `\n\n` (paragraph breaks) first. Any resulting piece that's still bigger than `chunk_size` gets recursively split on `. ` (sentence breaks), and if a single sentence is *still* too big, on raw characters -- the same `chunk_fixed_size` fallback from Step 3. Pieces smaller than `chunk_size` get greedily combined with their neighbors up to the limit, so this doesn't just split -- it also merges small pieces back together.

In [ ]:
def recursive_chunk(text, size, separators=None):
    """Split on the largest available separator first; recurse on oversized pieces with the next separator."""
    if separators is None:
        separators = ["\n\n", ". ", " "]
    separator, remaining_separators = separators[0], separators[1:]
    segments = [s for s in text.split(separator) if s.strip()]

    chunks = []
    current = ""
    for seg in segments:
        candidate = (current + separator + seg) if current else seg
        if len(candidate) <= size:
            current = candidate
        else:
            if current:
                chunks.append(current)
            if len(seg) > size:
                if remaining_separators:
                    chunks.extend(recursive_chunk(seg, size, remaining_separators))
                else:
                    chunks.extend(chunk_fixed_size(seg, size))
                current = ""
            else:
                current = seg
    if current:
        chunks.append(current)
    return chunks


chunks_recursive = recursive_chunk(manual_text, chunk_size)
show_chunks(chunks_recursive, "RECURSIVE")

### Step 6 — Heading-based chunking (a structural proxy for "semantic")

Split purely on `## ` heading markers, one chunk per section, regardless of how long or short that section is. This is the structural stand-in for semantic chunking described in the Conceptual Explanation -- it uses the document's own headings as topic boundaries instead of measuring sentence-to-sentence similarity with an embedding model.

In [ ]:
def chunk_by_section(text, heading_marker="## "):
    """One chunk per heading-delimited section. A structural proxy for semantic chunking -- no embeddings involved."""
    parts = text.split(heading_marker)
    return [heading_marker + part.strip() for part in parts[1:]]  # parts[0] is the title line before the first heading


chunks_by_section = chunk_by_section(manual_text)
show_chunks(chunks_by_section, "BY SECTION")

### Step 7 — Compare all four strategies side by side

Same source text, four strategies, four different chunk counts and size distributions. `avg_chars` alone can hide a lot -- `min_chars` and `max_chars` show how uneven each strategy's output actually is.

In [ ]:
strategy_results = {
    f"fixed_{chunk_size}": chunks_fixed,
    f"fixed_{chunk_size}_overlap_{chunk_overlap}": chunks_overlap,
    f"recursive_{chunk_size}": chunks_recursive,
    "by_section": chunks_by_section,
}

comparison_rows = []
for strategy, chunks in strategy_results.items():
    lengths = [len(c) for c in chunks]
    comparison_rows.append((
        strategy,
        len(chunks),
        round(sum(lengths) / len(lengths), 1),
        min(lengths),
        max(lengths),
    ))

comparison_df = spark.createDataFrame(
    comparison_rows, ["strategy", "num_chunks", "avg_chars", "min_chars", "max_chars"]
)
display(comparison_df)

### Step 8 — Apply recursive chunking to the 15 Notebook 02 documents

Recursive chunking is the strategy carried forward: it respects structure where structure exists and still has a safe fallback where it doesn't, which the other three strategies either lack (fixed-size, overlap) or can't guarantee bounded chunk sizes for (by-section). Each of Notebook 02's 15 documents is chunked independently and the results are written to a new `document_chunks` Delta table -- `doc_id` and `chunk_id` together identify a chunk, and `strategy` records which parameters produced it, in case this cell is re-run later with a different `chunk_size` widget value.

In [ ]:
documents_df = spark.table(documents_table)

chunk_rows = []
for row in documents_df.collect():
    doc_chunks = recursive_chunk(row["content"], chunk_size)
    for chunk_id, chunk_text in enumerate(doc_chunks):
        chunk_rows.append((
            row["doc_id"], row["title"], row["category"],
            chunk_id, chunk_text, len(chunk_text), f"recursive_{chunk_size}",
        ))

chunk_columns = ["doc_id", "title", "category", "chunk_id", "chunk_text", "char_count", "strategy"]
document_chunks_df = spark.createDataFrame(chunk_rows, chunk_columns)

document_chunks_df.write.mode("overwrite").saveAsTable(chunks_table)
print(f"Wrote {document_chunks_df.count()} chunks from {documents_df.count()} documents to {chunks_table}")

display(document_chunks_df.orderBy("doc_id", "chunk_id"))

## Inspect the Output

- In Step 3's boundary print, was a word actually cut in half? Which one?
- In Step 4, does the printed tail of chunk 0 exactly equal the head of chunk 1? What does that shared text actually buy you, given that the *same* sentence was already cut at that boundary in Step 3?
- Compare Step 5's chunk boundaries to Step 3's. Do any recursive chunks still end mid-sentence? If so, which ones, and why (check whether that chunk is close to `chunk_size`)?
- Step 5 also produced several very short, heading-only chunks (for example `## Cash Handling` alone, under 40 characters). Are those useful retrieval units on their own, or wasted rows? What would you change in `recursive_chunk` to avoid producing them?
- Look at Step 7's comparison table. How much do `by_section`'s chunk sizes vary? Would a single fixed `chunk_size` safely contain its largest section?
- In Step 8's `document_chunks` table, how many of the 15 documents produced more than one chunk? Given each source document is roughly 500-700 characters, does that match what you'd predict from `chunk_size=400`?

## Experimentation Section

1. Change the `chunk_size` widget to `200` and re-run Steps 3-7. Does recursive chunking start producing more heading-only orphan chunks, or does it start falling back to the character-level split inside single sentences?
2. Change `chunk_size` to `1000` and re-run. Does `chunk_by_section` (Step 6) now produce more total chunks than `recursive_chunk` (Step 5)? What does that tell you about when structural chunking beats size-based chunking?
3. Change `chunk_overlap` to `0`, re-run Step 4, and confirm the output is now identical to Step 3's. Then set `chunk_overlap` to `200` (half of `chunk_size=400`) -- how much does the chunk count grow, and is that a reasonable cost for the context it preserves?
4. Add `"\n"` as an extra separator level between `"\n\n"` and `". "` in `recursive_chunk`'s call in Step 5, and see whether it changes how the manual's sections get split.
5. Re-run Step 8 with `chunk_size` lowered to `150`. Notebook 02's documents are single paragraphs with no internal `\n\n` -- since there's no paragraph boundary to split on, where does the split actually happen, and does the resulting `document_chunks` table look meaningfully different from `chunk_size=400`'s?
6. Pick one chunk from Step 8's table and read it in isolation, without the rest of its source document. Is it understandable on its own? If not, that's exactly the problem Notebook 08's "enriched chunks" idea addresses.

## Common Errors / Limitations

- **No embedding-based semantic chunking here** -- Step 6 is a structural proxy (splits on headings), not real semantic chunking. True semantic chunking needs an embedding model to measure similarity between sentences, which this project doesn't introduce until `09_embeddings_basics.ipynb`; revisit Step 6 after that notebook if you want to implement the real thing.
- **Character counts are not token counts** -- embedding models and LLMs have limits measured in tokens, not characters (roughly 1 token per 4 characters for English text, but this varies by tokenizer). A chunk that looks safely small here in characters could still be too large in tokens; treat `chunk_size` in this notebook as an approximation, not a token budget.
- **Recursive chunking can still isolate near-empty chunks** -- as seen in Step 5, a heading that doesn't fit alongside its following paragraph becomes its own tiny chunk. This is a real limitation of the simple version implemented here, not a bug -- production splitters often add a final pass that merges undersized neighboring chunks, which this notebook does not implement.
- **Overlap mitigates bad boundaries, it doesn't fix them** -- overlap increases the chance a cut sentence appears whole somewhere, but does not guarantee it, and it multiplies storage and embedding cost by roughly `chunk_size / (chunk_size - chunk_overlap)`.
- **This corpus is small and clean** -- 15 short documents plus one hand-written multi-section manual. Real documents parsed with `ai_parse_document` (Notebook 04) carry tables, page breaks, and OCR artifacts that make chunk boundaries far messier than anything shown here.
- **`document_chunks` is overwritten on every run** -- Step 8 uses `.mode("overwrite")`, so re-running this notebook with a different `chunk_size` widget value replaces the entire table rather than versioning it. To compare strategies side by side later, write to a differently named table instead of overwriting this one.

## Summary

You implemented four chunking strategies in plain Python -- fixed-size, fixed-size with overlap, recursive/structure-aware, and a heading-based semantic proxy -- and watched each one produce a different set of chunks from the exact same source text. Fixed-size chunking is simple but blind to structure and can cut a sentence or a number in half; overlap reduces the damage from that blindness at a storage cost; recursive chunking respects paragraph and sentence boundaries first and only falls back to a hard character cut when it has to; and heading-based chunking approximates "semantic" boundaries using the document's own structure, standing in for the embedding-based version until Notebook 09. You then applied the recursive strategy across all 15 Notebook 02 documents and wrote the result to a `document_chunks` table -- the raw material Notebook 08 will enrich and prepare for retrieval.

## Suggested Exercises

- Pick any one of Notebook 02's 15 original documents, read its full content, and manually decide where *you* would place chunk boundaries. Compare your answer to what `recursive_chunk` actually produced for that document in Step 8.
- Compute the average chunk length across the entire `document_chunks` table (a two-line `groupBy`/`agg` query) and compare it to the `chunk_size` widget value -- are most chunks close to the limit, or well under it? What does that tell you about how much of the corpus is single-paragraph documents versus multi-paragraph ones?
- When you're ready, move on to **`08_enriched_chunks_and_search_preparation.ipynb`**, which takes the `document_chunks` table produced here and asks: what should actually be embedded (`chunk_to_embed`), what should actually be shown to the LLM (`chunk_to_retrieve`), and how does `ai_prep_search` fit between the two?